In [18]:
!pip install kaggle nltk scikit-learn pandas --quiet

!pip install nltk scikit-learn pandas flask pyngrok --quiet

import re, os, json, nltk
import pandas as pd
from collections import Counter
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

for pkg in ['punkt','punkt_tab','stopwords','averaged_perceptron_tagger',
            'averaged_perceptron_tagger_eng','vader_lexicon']:
    nltk.download(pkg, quiet=True)

print('✅ Setup complete')

✅ Setup complete


In [19]:
import zipfile

# Extract
with zipfile.ZipFile('/content/amazon-product-reviews.zip', 'r') as z:
    z.extractall('/content/amazon_reviews/')

# Find CSV
all_files = []
for root, dirs, files in os.walk('/content/amazon_reviews/'):
    for f in files:
        if f.endswith('.csv'):
            all_files.append(os.path.join(root, f))

df_raw = pd.read_csv(all_files[0])
print('✅ Dataset loaded!')
print(f'Shape: {df_raw.shape}')
print(f'Columns: {list(df_raw.columns)}')

✅ Dataset loaded!
Shape: (568454, 10)
Columns: ['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text']


In [20]:
df = df_raw[['ProductId','Score','Summary','Text']].copy()
df = df.dropna(subset=['Text','Score'])

def map_sentiment(score):
    if score >= 4: return 'Positive'
    elif score <= 2: return 'Negative'
    else: return 'Neutral'

df['sentiment_label'] = df['Score'].apply(map_sentiment)

pos = df[df['sentiment_label']=='Positive'].sample(1000, random_state=42)
neg = df[df['sentiment_label']=='Negative'].sample(1000, random_state=42)
neu = df[df['sentiment_label']=='Neutral'].sample(1000, random_state=42)
df  = pd.concat([pos,neg,neu]).sample(frac=1, random_state=42).reset_index(drop=True)

stop_words = set(stopwords.words('english'))
stemmer    = PorterStemmer()

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'<.*?>','',text)
    text = re.sub(r'http\S+|www\S+','',text)
    text = re.sub(r'[^a-zA-Z\s]','',text)
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]
    tokens = [stemmer.stem(w) for w in tokens]
    return ' '.join(tokens)

df['clean_text'] = df['Text'].apply(preprocess_text)
print(f'✅ Data ready — {len(df)} reviews')
print(df['sentiment_label'].value_counts())

✅ Data ready — 3000 reviews
sentiment_label
Negative    1000
Positive    1000
Neutral     1000
Name: count, dtype: int64


In [21]:
label_map = {'Positive':1,'Negative':0,'Neutral':2}
df['label_num'] = df['sentiment_label'].map(label_map)

X = df['clean_text']
y = df['label_num']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

vectorizer    = CountVectorizer(max_features=5000)
X_train_vec   = vectorizer.fit_transform(X_train)
X_test_vec    = vectorizer.transform(X_test)

model = MultinomialNB()
model.fit(X_train_vec, y_train)

y_pred   = model.predict(X_test_vec)
accuracy = accuracy_score(y_test, y_pred)

print(f'✅ Model trained — Accuracy: {accuracy:.4f}')
print(classification_report(y_test, y_pred, target_names=['Negative','Positive','Neutral']))

✅ Model trained — Accuracy: 0.6200
              precision    recall  f1-score   support

    Negative       0.60      0.61      0.60       183
    Positive       0.81      0.64      0.72       232
     Neutral       0.48      0.61      0.54       185

    accuracy                           0.62       600
   macro avg       0.63      0.62      0.62       600
weighted avg       0.65      0.62      0.63       600



In [22]:
sia = SentimentIntensityAnalyzer()

def get_sentiment(text):
    score = sia.polarity_scores(str(text))['compound']
    if score >= 0.05:   return 'Positive', round(score,3)
    elif score <= -0.05: return 'Negative', round(score,3)
    else:                return 'Neutral',  round(score,3)

aspect_keywords = {
    'quality':   ['quality','durable','build','material','sturdy','cheap','flimsy'],
    'taste':     ['taste','flavor','delicious','yummy','bland','sweet','fresh','stale'],
    'packaging': ['packaging','pack','box','sealed','damaged','container','wrapper'],
    'price':     ['price','cost','value','worth','expensive','affordable','overpriced'],
    'delivery':  ['delivery','shipping','arrived','fast','slow','late','quick'],
    'smell':     ['smell','odor','fragrance','scent','aroma','stink'],
    'texture':   ['texture','smooth','rough','soft','hard','crispy','chewy'],
    'service':   ['service','support','customer','return','refund','helpful','rude'],
}
positive_words = set(['good','great','excellent','amazing','fantastic','outstanding',
                      'perfect','best','superb','brilliant','wonderful','delicious',
                      'fresh','helpful','fast','smooth','sturdy','affordable','love',
                      'loved','recommend','crispy','tasty'])
negative_words = set(['bad','terrible','worst','horrible','poor','disappointing',
                      'slow','cheap','flimsy','broken','damaged','stale','bland',
                      'awful','rude','expensive','overpriced','waste','avoid',
                      'disgusting','smells','leak'])

def extract_aspects(text):
    text  = str(text)
    tlow  = text.lower()
    pros, cons = [], []
    for aspect, keywords in aspect_keywords.items():
        for sent in sent_tokenize(text):
            sl = sent.lower()
            if any(kw in sl for kw in keywords):
                words = set(sl.split())
                if any(pw in words for pw in positive_words):
                    pros.append(aspect.capitalize()); break
                elif any(nw in words for nw in negative_words):
                    cons.append(aspect.capitalize()); break
    return list(set(pros)), list(set(cons))

df['sentiment'], df['score'] = zip(*df['Text'].apply(get_sentiment))
df['pros'],      df['cons']  = zip(*df['Text'].apply(extract_aspects))

print('✅ Sentiment + aspects done')
print(df[['sentiment','score','pros','cons']].head())

✅ Sentiment + aspects done
  sentiment  score         pros cons
0  Positive  0.236           []   []
1  Positive  0.475           []   []
2  Positive  0.751  [Packaging]   []
3  Positive  0.840           []   []
4  Positive  0.318           []   []


In [25]:
top_products  = df['ProductId'].value_counts().head(8).index.tolist()
summary_data  = []

for pid in top_products:
    pdf      = df[df['ProductId']==pid]
    pos      = len(pdf[pdf['sentiment']=='Positive'])
    neg      = len(pdf[pdf['sentiment']=='Negative'])
    neu      = len(pdf[pdf['sentiment']=='Neutral'])
    avg      = round(pdf['score'].mean(),3)
    all_pros = [p for sub in pdf['pros'] for p in sub]
    all_cons = [c for sub in pdf['cons'] for c in sub]
    top_pros = [x for x,_ in Counter(all_pros).most_common(3)]
    top_cons = [x for x,_ in Counter(all_cons).most_common(3)]
    verdict  = 'Recommended' if avg>=0.2 else ('Not Recommended' if avg<=-0.2 else 'Mixed')
    sample_r = pdf[['Text','sentiment','score']].head(3).to_dict('records')
    summary_data.append({
        'product': pid,
        'total': len(pdf), 'pos': pos, 'neg': neg, 'neu': neu,
        'avg': avg, 'verdict': verdict,
        'pros': top_pros, 'cons': top_cons,
        'reviews': [{'text': r['Text'][:150]+'...','sentiment':r['sentiment'],'score':r['score']} for r in sample_r]
    })

total_pos  = len(df[df['sentiment']=='Positive'])
total_neg  = len(df[df['sentiment']=='Negative'])
total_neu  = len(df[df['sentiment']=='Neutral'])
acc_pct    = round(accuracy*100, 2)

print(f'✅ Summary data built for {len(summary_data)} products')

✅ Summary data built for 8 products


In [26]:
from flask import Flask, request, jsonify
from pyngrok import ngrok
import threading, time, json

app = Flask(__name__)

HTML_PAGE = """
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1.0">
<title>NLP Product Sentiment Analyzer</title>
<link href="https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=DM+Sans:wght@300;400;500;600&display=swap" rel="stylesheet">
<style>
*{box-sizing:border-box;margin:0;padding:0;}
:root{
  --green:#1D9E75;--green-l:#E1F5EE;--green-d:#085041;
  --red:#D85A30;--red-l:#FAECE7;--red-d:#4A1B0C;
  --amber:#EF9F27;--amber-l:#FAEEDA;
  --blue:#378ADD;--blue-l:#E6F1FB;
  --bg:#f4f4f0;--card:#fff;--txt:#1a1a18;--txt2:#6b6b67;
  --border:rgba(0,0,0,0.1);--r:12px;
  --mono:'Space Mono',monospace;--sans:'DM Sans',sans-serif;
}
body{font-family:var(--sans);background:var(--bg);color:var(--txt);}
nav{background:var(--card);border-bottom:1px solid var(--border);padding:0 2rem;display:flex;align-items:center;justify-content:space-between;height:56px;position:sticky;top:0;z-index:50;}
.nav-logo{font-size:15px;font-weight:600;}
.nav-logo em{font-style:normal;color:var(--green);}
.nav-links{display:flex;gap:4px;}
.nav-link{padding:6px 14px;font-size:13px;font-weight:500;color:var(--txt2);background:none;border:none;cursor:pointer;border-radius:8px;font-family:var(--sans);transition:all .2s;}
.nav-link:hover{background:var(--bg);color:var(--txt);}
.nav-link.active{background:var(--green-l);color:var(--green-d);}
.nav-badge{font-size:10px;font-family:var(--mono);background:var(--green-l);color:var(--green-d);padding:3px 8px;border-radius:20px;border:1px solid #9FE1CB;}
.hero{background:var(--card);border-bottom:1px solid var(--border);padding:3rem 2rem 2.5rem;}
.hero-inner{max-width:900px;margin:0 auto;}
.badge{display:inline-flex;align-items:center;gap:6px;background:var(--green-l);color:var(--green-d);font-size:11px;font-family:var(--mono);padding:4px 12px;border-radius:20px;margin-bottom:1rem;border:1px solid #9FE1CB;}
.badge span{width:6px;height:6px;border-radius:50%;background:var(--green);animation:pulse 2s infinite;}
@keyframes pulse{0%,100%{opacity:1;}50%{opacity:0.3;}}
.hero h1{font-size:32px;font-weight:600;line-height:1.2;margin-bottom:10px;}
.hero h1 em{font-style:normal;color:var(--green);}
.hero p{font-size:15px;color:var(--txt2);line-height:1.7;max-width:600px;}
.hero-stats{display:flex;gap:2rem;margin-top:1.5rem;flex-wrap:wrap;}
.hero-stat{display:flex;flex-direction:column;}
.hero-stat-val{font-size:24px;font-weight:600;font-family:var(--mono);color:var(--green);}
.hero-stat-label{font-size:12px;color:var(--txt2);margin-top:2px;}
.main{max-width:940px;margin:0 auto;padding:2rem 1.5rem;}
.page{display:none;}.page.active{display:block;}
.stats{display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin-bottom:1.5rem;}
.stat{background:var(--card);border-radius:var(--r);padding:16px 18px;border:1px solid var(--border);}
.stat-label{font-size:11px;color:var(--txt2);text-transform:uppercase;letter-spacing:.07em;margin-bottom:6px;}
.stat-val{font-size:24px;font-weight:600;font-family:var(--mono);}
.g{color:var(--green);}.r{color:var(--red);}.a{color:var(--amber);}
.stat-sub{font-size:11px;color:var(--txt2);margin-top:3px;}
.section-title{font-size:13px;font-weight:500;color:var(--txt2);text-transform:uppercase;letter-spacing:.08em;margin-bottom:1rem;}
.grid{display:grid;grid-template-columns:1fr 1fr;gap:1rem;}
.pcard{background:var(--card);border-radius:var(--r);border:1px solid var(--border);padding:1.25rem;cursor:pointer;transition:all .2s;}
.pcard:hover{border-color:var(--green);transform:translateY(-2px);box-shadow:0 4px 20px rgba(29,158,117,0.1);}
.pcard-top{display:flex;justify-content:space-between;align-items:flex-start;margin-bottom:1rem;}
.pcard-id{font-size:12px;font-family:var(--mono);font-weight:600;word-break:break-all;color:var(--txt);}
.pcard-count{font-size:11px;color:var(--txt2);margin-top:2px;}
.verdict{font-size:11px;padding:3px 10px;border-radius:20px;font-weight:600;flex-shrink:0;margin-left:8px;}
.verdict.rec{background:var(--green-l);color:var(--green-d);}
.verdict.not{background:var(--red-l);color:var(--red-d);}
.verdict.mix{background:var(--amber-l);color:#633806;}
.sent-labels{display:flex;justify-content:space-between;font-size:11px;color:var(--txt2);margin-bottom:4px;}
.sent-bar{height:8px;border-radius:4px;display:flex;overflow:hidden;background:#eee;margin-bottom:10px;}
.seg-p{background:var(--green);}.seg-n{background:var(--red);}.seg-u{background:#B4B2A9;}
.score-row{font-size:12px;color:var(--txt2);margin-bottom:10px;}
.score-num{font-family:var(--mono);font-weight:600;font-size:13px;}
.tag-label{font-size:10px;color:var(--txt2);text-transform:uppercase;letter-spacing:.06em;margin-bottom:4px;}
.tags{display:flex;flex-wrap:wrap;gap:4px;margin-bottom:8px;}
.tag{font-size:11px;padding:3px 9px;border-radius:20px;font-weight:500;}
.tag.pro{background:var(--green-l);color:var(--green-d);}
.tag.con{background:var(--red-l);color:var(--red-d);}
.click-hint{font-size:11px;color:var(--txt2);margin-top:8px;text-align:right;}
.filter-bar{display:flex;gap:8px;flex-wrap:wrap;margin-bottom:1.25rem;align-items:center;}
.filter-label{font-size:13px;color:var(--txt2);}
.fbtn{padding:6px 14px;border-radius:8px;font-size:12px;font-weight:500;font-family:var(--sans);cursor:pointer;border:1px solid var(--border);background:var(--card);color:var(--txt2);transition:all .18s;}
.fbtn:hover{border-color:var(--green);color:var(--green);}
.fbtn.active{background:var(--green-l);color:var(--green-d);border-color:#9FE1CB;}
.demo-card{background:var(--card);border:1px solid var(--border);border-radius:var(--r);padding:1.5rem;margin-bottom:1rem;}
.demo-card h3{font-size:14px;font-weight:500;margin-bottom:4px;}
.demo-card p{font-size:13px;color:var(--txt2);margin-bottom:1rem;line-height:1.6;}
textarea{width:100%;resize:none;font-family:var(--sans);font-size:14px;background:#f7f7f5;border:1px solid rgba(0,0,0,0.15);border-radius:8px;padding:12px;color:var(--txt);outline:none;line-height:1.6;transition:border-color .2s;}
textarea:focus{border-color:var(--green);}
.btn-row{display:flex;gap:8px;margin-top:12px;flex-wrap:wrap;}
.btn{padding:9px 20px;border-radius:8px;font-size:13px;font-weight:500;font-family:var(--sans);cursor:pointer;border:1px solid rgba(0,0,0,0.15);background:transparent;color:var(--txt);transition:all .18s;}
.btn:hover{background:#f0f0ec;}
.btn.primary{background:var(--green);color:#fff;border-color:var(--green);}
.btn.primary:hover{background:var(--green-d);}
.btn:disabled{opacity:0.6;cursor:not-allowed;}
.result-box{background:#f7f7f5;border-radius:var(--r);border:1px solid var(--border);padding:1.25rem;margin-top:1rem;display:none;}
.result-box.show{display:block;animation:fadeUp .3s ease;}
@keyframes fadeUp{from{opacity:0;transform:translateY(8px);}to{opacity:1;transform:none;}}
.res-header{display:flex;align-items:center;gap:12px;margin-bottom:12px;flex-wrap:wrap;}
.res-badge{display:inline-flex;align-items:center;gap:6px;padding:6px 16px;border-radius:20px;font-size:14px;font-weight:600;}
.res-badge.pos{background:var(--green-l);color:var(--green-d);}
.res-badge.neg{background:var(--red-l);color:var(--red-d);}
.res-badge.neu{background:var(--amber-l);color:#633806;}
.conf-track{flex:1;min-width:120px;height:5px;background:var(--border);border-radius:3px;overflow:hidden;}
.conf-fill{height:100%;border-radius:3px;transition:width .7s ease;}
.trace{margin-top:12px;}
.trace-row{display:flex;align-items:flex-start;gap:12px;padding:6px 0;border-bottom:1px solid var(--border);font-size:13px;}
.trace-row:last-child{border-bottom:none;}
.tlabel{min-width:100px;color:var(--txt2);font-family:var(--mono);font-size:11px;padding-top:2px;}
.tval{flex:1;color:var(--txt);line-height:1.6;}
.tval code{background:#efefeb;padding:2px 6px;border-radius:4px;font-family:var(--mono);font-size:11px;color:var(--green);}
.samples{display:flex;gap:6px;margin-bottom:1rem;flex-wrap:wrap;}
.sample-btn{font-size:12px;padding:5px 12px;border-radius:8px;border:1px solid var(--border);background:var(--card);cursor:pointer;color:var(--txt2);font-family:var(--sans);transition:all .18s;}
.sample-btn:hover{border-color:var(--green);color:var(--green);}
.pipeline{display:flex;overflow-x:auto;margin-bottom:1rem;padding-bottom:4px;}
.pipe-step{flex:1;min-width:95px;background:var(--card);border:1px solid var(--border);padding:14px 8px 12px;text-align:center;cursor:pointer;transition:background .2s;}
.pipe-step:first-child{border-radius:var(--r) 0 0 var(--r);}
.pipe-step:last-child{border-radius:0 var(--r) var(--r) 0;}
.pipe-step+.pipe-step{border-left:none;}
.pipe-step:hover{background:#f0faf6;}
.pipe-step.active{background:var(--green-l);border-color:#9FE1CB;}
.pipe-icon{font-size:20px;margin-bottom:5px;}
.pipe-name{font-size:11px;font-weight:500;color:var(--txt);}
.pipe-tag{font-size:10px;color:var(--txt2);font-family:var(--mono);margin-top:2px;}
.pipe-step.active .pipe-name{color:var(--green-d);}
.pipe-step.active .pipe-tag{color:var(--green);}
.step-detail{background:var(--card);border:1px solid var(--border);border-radius:var(--r);padding:1.25rem;}
.step-title{font-size:15px;font-weight:500;margin-bottom:8px;}
.step-desc{font-size:13px;color:var(--txt2);line-height:1.7;margin-bottom:12px;}
.step-ex{background:#f7f7f5;border-radius:8px;padding:12px;font-size:12px;font-family:var(--mono);line-height:1.8;white-space:pre-wrap;word-break:break-word;border:1px solid var(--border);}
.overlay{position:fixed;top:0;left:0;width:100%;height:100%;background:rgba(0,0,0,0.45);display:none;align-items:center;justify-content:center;z-index:200;padding:1rem;}
.overlay.show{display:flex;}
.modal{background:var(--card);border-radius:var(--r);padding:1.5rem;max-width:580px;width:100%;max-height:85vh;overflow-y:auto;}
.modal-header{display:flex;justify-content:space-between;align-items:center;margin-bottom:1.25rem;}
.modal-title{font-size:14px;font-weight:600;font-family:var(--mono);word-break:break-all;}
.close-btn{background:none;border:none;font-size:20px;cursor:pointer;color:var(--txt2);line-height:1;}
.rev-item{padding:10px 0;border-bottom:1px solid var(--border);font-size:13px;line-height:1.6;}
.rev-item:last-child{border-bottom:none;}
.rev-badge{display:inline-block;font-size:10px;padding:2px 8px;border-radius:10px;font-weight:600;margin-bottom:5px;}
.rev-badge.pos{background:var(--green-l);color:var(--green-d);}
.rev-badge.neg{background:var(--red-l);color:var(--red-d);}
.rev-badge.neu{background:var(--amber-l);color:#633806;}
.about-grid{display:grid;grid-template-columns:1fr 1fr;gap:1rem;}
.about-card{background:var(--card);border:1px solid var(--border);border-radius:var(--r);padding:1.25rem;}
.about-card h3{font-size:11px;font-weight:500;color:var(--txt2);text-transform:uppercase;letter-spacing:.08em;margin-bottom:1rem;}
.about-card p{font-size:13px;line-height:1.7;color:var(--txt);}
.team-list{list-style:none;}
.team-list li{display:flex;align-items:center;gap:10px;padding:8px 0;border-bottom:1px solid var(--border);font-size:13px;}
.team-list li:last-child{border-bottom:none;}
.avatar{width:30px;height:30px;border-radius:50%;background:var(--green-l);color:var(--green-d);display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:600;flex-shrink:0;}
.team-name{font-weight:500;}
.team-reg{font-size:11px;color:var(--txt2);font-family:var(--mono);}
.tech-list{list-style:none;}
.tech-list li{display:flex;align-items:center;gap:8px;padding:6px 0;border-bottom:1px solid var(--border);font-size:13px;}
.tech-list li:last-child{border-bottom:none;}
.tech-dot{width:6px;height:6px;border-radius:50%;background:var(--green);flex-shrink:0;}
.lim-item{display:flex;gap:8px;padding:6px 0;font-size:13px;line-height:1.6;border-bottom:1px solid var(--border);}
.lim-item:last-child{border-bottom:none;}
.lim-icon{flex-shrink:0;margin-top:1px;}
footer{text-align:center;padding:2rem;font-size:12px;color:var(--txt2);border-top:1px solid var(--border);margin-top:2rem;}
@media(max-width:640px){
  .stats{grid-template-columns:1fr 1fr;}
  .grid{grid-template-columns:1fr;}
  .about-grid{grid-template-columns:1fr;}
  .hero h1{font-size:24px;}
  nav{padding:0 1rem;}
  .main{padding:1.5rem 1rem;}
}
</style>
</head>
<body>

<nav>
  <div class="nav-logo">Review<em>Lens</em></div>
  <div class="nav-links">
    <button class="nav-link active" onclick="goTo('overview',this)">Overview</button>
    <button class="nav-link" onclick="goTo('products',this)">Products</button>
    <button class="nav-link" onclick="goTo('demo',this)">Demo</button>
    <button class="nav-link" onclick="goTo('pipeline',this)">Pipeline</button>
    <button class="nav-link" onclick="goTo('about',this)">About</button>
  </div>
  <div class="nav-badge">Kaggle Dataset</div>
</nav>

<div class="hero">
  <div class="hero-inner">
    <div class="badge"><span></span>NLP Mini Project &middot; CSA4028 &middot; VIT Bhopal</div>
    <h1>Product Review <em>Summarizer</em><br>&amp; Sentiment Analyzer</h1>
    <p>Real Amazon product reviews analyzed using VADER sentiment analysis, aspect extraction, and Naive Bayes classification. Built with NLTK and Scikit-learn.</p>
    <div class="hero-stats">
      <div class="hero-stat"><div class="hero-stat-val" id="hTotal">-</div><div class="hero-stat-label">Reviews analyzed</div></div>
      <div class="hero-stat"><div class="hero-stat-val" id="hAcc">-</div><div class="hero-stat-label">Model accuracy</div></div>
      <div class="hero-stat"><div class="hero-stat-val" id="hProducts">-</div><div class="hero-stat-label">Products tracked</div></div>
      <div class="hero-stat"><div class="hero-stat-val" id="hPos">-</div><div class="hero-stat-label">Positive rate</div></div>
    </div>
  </div>
</div>

<div class="main">
  <div class="page active" id="page-overview">
    <div class="stats" id="statsGrid"></div>
    <div class="section-title">Top products at a glance</div>
    <div class="grid" id="overviewGrid"></div>
  </div>

  <div class="page" id="page-products">
    <div class="filter-bar">
      <span class="filter-label">Filter:</span>
      <button class="fbtn active" onclick="filterV('All',this)">All</button>
      <button class="fbtn" onclick="filterV('Recommended',this)">Recommended</button>
      <button class="fbtn" onclick="filterV('Not Recommended',this)">Not Recommended</button>
      <button class="fbtn" onclick="filterV('Mixed',this)">Mixed</button>
    </div>
    <div class="grid" id="productGrid"></div>
  </div>

  <div class="page" id="page-demo">
    <div class="demo-card">
      <h3>Analyze any product review</h3>
      <p>Enter a review below and the NLP pipeline will detect sentiment, extract pros and cons, and show you the analysis step by step.</p>
      <div class="samples">
        <span style="font-size:12px;color:var(--txt2);padding:5px 0;">Try a sample:</span>
        <button class="sample-btn" onclick="setDemo(0)">Positive review</button>
        <button class="sample-btn" onclick="setDemo(1)">Negative review</button>
        <button class="sample-btn" onclick="setDemo(2)">Mixed review</button>
      </div>
      <textarea id="demoText" rows="5" placeholder="e.g. This product is absolutely amazing! Great quality and fast delivery..."></textarea>
      <div class="btn-row">
        <button class="btn primary" id="analyzeBtn" onclick="runDemo()">Analyze review</button>
        <button class="btn" onclick="document.getElementById('demoText').value=''">Clear</button>
      </div>
    </div>
    <div class="result-box" id="resultBox">
      <div class="res-header">
        <div class="res-badge" id="resBadge"></div>
        <div class="conf-track"><div class="conf-fill" id="confFill" style="width:0%"></div></div>
        <div id="confLabel" style="font-size:12px;color:var(--txt2);white-space:nowrap;"></div>
      </div>
      <div class="tags" id="resPros"></div>
      <div class="tags" id="resCons"></div>
      <div class="trace" id="resTrace"></div>
    </div>
  </div>

  <div class="page" id="page-pipeline">
    <p style="font-size:13px;color:var(--txt2);margin-bottom:1rem;line-height:1.6;">Click each stage to explore how raw text transforms through the NLP pipeline.</p>
    <div class="pipeline" id="pipelineBar"></div>
    <div class="step-detail">
      <div class="step-title" id="stepTitle"></div>
      <div class="step-desc" id="stepDesc"></div>
      <pre class="step-ex" id="stepEx"></pre>
    </div>
  </div>

  <div class="page" id="page-about">
    <div class="about-grid">
      <div class="about-card">
        <h3>Project overview</h3>
        <p>A hybrid NLP pipeline integrating rule-based preprocessing and machine learning for sentiment classification on real Amazon product reviews from Kaggle. Covers tokenization, morphological analysis, POS tagging, aspect extraction, and Bag-of-Words feature extraction.</p>
      </div>
      <div class="about-card">
        <h3>Technology stack</h3>
        <ul class="tech-list">
          <li><span class="tech-dot"></span>Python 3.x</li>
          <li><span class="tech-dot"></span>NLTK — tokenization, POS, VADER</li>
          <li><span class="tech-dot"></span>Scikit-learn — Naive Bayes, vectorizer</li>
          <li><span class="tech-dot"></span>Pandas — data manipulation</li>
          <li><span class="tech-dot"></span>Flask — web server</li>
          <li><span class="tech-dot"></span>Kaggle — Amazon reviews dataset</li>
        </ul>
      </div>
      <div class="about-card">
        <h3>Team members</h3>
        <ul class="team-list">
          <li><div class="avatar">AD</div><div><div class="team-name">Atharw Dwivedi</div><div class="team-reg">23BAI11373</div></div></li>
          <li><div class="avatar">AM</div><div><div class="team-name">Ayushman Mishra</div><div class="team-reg">23BAI11256</div></div></li>
          <li><div class="avatar">RT</div><div><div class="team-name">Rana Talukdar</div><div class="team-reg">23BAI10186</div></div></li>
          <li><div class="avatar">MK</div><div><div class="team-name">Mohammad Kaif</div><div class="team-reg">23BAI10510</div></div></li>
        </ul>
      </div>
      <div class="about-card">
        <h3>Limitations and future scope</h3>
        <div class="lim-item"><span class="lim-icon">warning</span><span>Bag-of-Words ignores word context and order</span></div>
        <div class="lim-item"><span class="lim-icon">warning</span><span>Cannot handle sarcasm or idiomatic expressions</span></div>
        <div class="lim-item"><span class="lim-icon">rocket</span><span>Future: Word2Vec and GloVe word embeddings</span></div>
        <div class="lim-item"><span class="lim-icon">rocket</span><span>Future: LSTM and Transformer models</span></div>
        <div class="lim-item"><span class="lim-icon">rocket</span><span>Future: Multi-language support</span></div>
      </div>
    </div>
  </div>
</div>

<footer>CSA4028 Natural Language Processing &nbsp;·&nbsp; VIT Bhopal &nbsp;·&nbsp; Winter Semester 2025-26</footer>

<div class="overlay" id="overlay" onclick="closeModal(event)">
  <div class="modal">
    <div class="modal-header">
      <div class="modal-title" id="modalTitle"></div>
      <button class="close-btn" onclick="document.getElementById('overlay').classList.remove('show')">X</button>
    </div>
    <div id="modalBody"></div>
  </div>
</div>

<script>
const DATA   = __SUMMARY_DATA__;
const TOTALS = __TOTALS__;
const ACC    = __ACCURACY__;

document.getElementById('hTotal').textContent    = TOTALS.total.toLocaleString();
document.getElementById('hAcc').textContent      = ACC + '%';
document.getElementById('hProducts').textContent = DATA.length;
document.getElementById('hPos').textContent      = Math.round(TOTALS.pos/TOTALS.total*100) + '%';

document.getElementById('statsGrid').innerHTML = `
  <div class="stat"><div class="stat-label">Total Reviews</div><div class="stat-val">${TOTALS.total.toLocaleString()}</div><div class="stat-sub">from Kaggle</div></div>
  <div class="stat"><div class="stat-label">Positive</div><div class="stat-val g">${TOTALS.pos.toLocaleString()}</div><div class="stat-sub">${Math.round(TOTALS.pos/TOTALS.total*100)}% of total</div></div>
  <div class="stat"><div class="stat-label">Negative</div><div class="stat-val r">${TOTALS.neg.toLocaleString()}</div><div class="stat-sub">${Math.round(TOTALS.neg/TOTALS.total*100)}% of total</div></div>
  <div class="stat"><div class="stat-label">Model Accuracy</div><div class="stat-val g">${ACC}%</div><div class="stat-sub">Naive Bayes</div></div>
`;

function makeCard(d){
  const pw = d.total>0?(d.pos/d.total*100).toFixed(0):0;
  const nw = d.total>0?(d.neg/d.total*100).toFixed(0):0;
  const uw = d.total>0?(d.neu/d.total*100).toFixed(0):0;
  const vc = d.verdict==='Recommended'?'rec':d.verdict==='Not Recommended'?'not':'mix';
  const vi = d.verdict==='Recommended'?'Yes':d.verdict==='Not Recommended'?'No':'Mixed';
  const sc = d.avg>=0.05?'var(--green)':d.avg<=-0.05?'var(--red)':'var(--amber)';
  const pH = d.pros.length?d.pros.map(p=>`<span class="tag pro">+ ${p}</span>`).join(''):'<span style="font-size:11px;color:var(--txt2);">None detected</span>';
  const cH = d.cons.length?d.cons.map(c=>`<span class="tag con">- ${c}</span>`).join(''):'<span style="font-size:11px;color:var(--txt2);">None detected</span>';
  return `
    <div class="pcard" onclick="openModal('${d.product}')" data-verdict="${d.verdict}">
      <div class="pcard-top">
        <div><div class="pcard-id">${d.product.substring(0,20)}...</div><div class="pcard-count">${d.total} reviews</div></div>
        <span class="verdict ${vc}">${vi}</span>
      </div>
      <div class="sent-labels"><span>Pos ${d.pos}</span><span>Neu ${d.neu}</span><span>Neg ${d.neg}</span></div>
      <div class="sent-bar"><div class="seg-p" style="width:${pw}%"></div><div class="seg-u" style="width:${uw}%"></div><div class="seg-n" style="width:${nw}%"></div></div>
      <div class="score-row">Avg score: <span class="score-num" style="color:${sc}">${d.avg>=0?'+':''}${d.avg}</span></div>
      <div class="tag-label">Top pros</div><div class="tags">${pH}</div>
      <div class="tag-label">Top cons</div><div class="tags">${cH}</div>
      <div class="click-hint">Click for sample reviews</div>
    </div>`;
}

document.getElementById('overviewGrid').innerHTML = DATA.map(makeCard).join('');
document.getElementById('productGrid').innerHTML  = DATA.map(makeCard).join('');

function goTo(name,el){
  document.querySelectorAll('.nav-link').forEach(l=>l.classList.remove('active'));
  document.querySelectorAll('.page').forEach(p=>p.classList.remove('active'));
  el.classList.add('active');
  document.getElementById('page-'+name).classList.add('active');
  window.scrollTo({top:0,behavior:'smooth'});
}

function filterV(v,el){
  document.querySelectorAll('.fbtn').forEach(b=>b.classList.remove('active'));
  el.classList.add('active');
  const filtered = v==='All'?DATA:DATA.filter(d=>d.verdict===v);
  document.getElementById('productGrid').innerHTML = filtered.length
    ? filtered.map(makeCard).join('')
    : '<p style="font-size:13px;color:var(--txt2);padding:1rem;">No products found.</p>';
}

function openModal(pid){
  const d = DATA.find(x=>x.product===pid);
  if(!d) return;
  document.getElementById('modalTitle').textContent = pid;
  const pw = d.total>0?(d.pos/d.total*100).toFixed(0):0;
  const nw = d.total>0?(d.neg/d.total*100).toFixed(0):0;
  const uw = d.total>0?(d.neu/d.total*100).toFixed(0):0;
  const sc = d.avg>=0.05?'var(--green)':d.avg<=-0.05?'var(--red)':'var(--amber)';
  const vc = d.verdict==='Recommended'?'rec':d.verdict==='Not Recommended'?'not':'mix';
  const revsH = d.reviews.map(r=>{
    const bc = r.sentiment==='Positive'?'pos':r.sentiment==='Negative'?'neg':'neu';
    return '<div class="rev-item"><span class="rev-badge '+bc+'">'+r.sentiment+' ('+r.score+')</span><br>'+r.text+'</div>';
  }).join('');
  document.getElementById('modalBody').innerHTML =
    '<div class="sent-labels"><span>Pos '+d.pos+'</span><span>Neu '+d.neu+'</span><span>Neg '+d.neg+'</span></div>'+
    '<div class="sent-bar" style="margin:8px 0 12px;"><div class="seg-p" style="width:'+pw+'%"></div><div class="seg-u" style="width:'+uw+'%"></div><div class="seg-n" style="width:'+nw+'%"></div></div>'+
    '<div style="font-size:12px;color:var(--txt2);margin-bottom:12px;">Avg score: <span style="font-family:var(--mono);font-weight:600;color:'+sc+';">'+(d.avg>=0?'+':'')+d.avg+'</span></div>'+
    '<div class="tag-label">Pros</div><div class="tags" style="margin-bottom:8px;">'+(d.pros.map(p=>'<span class="tag pro">'+p+'</span>').join('')||'None')+'</div>'+
    '<div class="tag-label">Cons</div><div class="tags" style="margin-bottom:16px;">'+(d.cons.map(c=>'<span class="tag con">'+c+'</span>').join('')||'None')+'</div>'+
    '<div class="section-title">Sample reviews</div>'+revsH;
  document.getElementById('overlay').classList.add('show');
}

function closeModal(e){
  if(e.target===document.getElementById('overlay'))
    document.getElementById('overlay').classList.remove('show');
}

const DEMOS=[
  'This product is absolutely amazing! The quality is excellent and delivery was super fast. Taste is delicious and fresh. Highly recommend to everyone!',
  'Terrible product. Arrived completely damaged and the smell was horrible. Very poor quality and packaging was broken. Total waste of money. Avoid!',
  'The taste is good and quality is decent overall. Packaging could be better. Price is slightly expensive but delivery was fast. Acceptable product.'
];
const posW=['good','great','excellent','amazing','fantastic','outstanding','perfect','best','superb','brilliant','wonderful','delicious','fresh','helpful','fast','smooth','affordable','love','recommend','crispy','tasty','highly'];
const negW=['bad','terrible','worst','horrible','poor','disappointing','slow','cheap','flimsy','broken','damaged','stale','bland','awful','rude','expensive','waste','avoid','disgusting','smells','leak','overpriced'];
const ASPECTS={Quality:['quality','durable','build','material','sturdy'],Taste:['taste','flavor','delicious','yummy','bland','sweet','fresh','stale'],Packaging:['packaging','pack','box','sealed','damaged','container'],Price:['price','cost','value','worth','expensive','affordable'],Delivery:['delivery','shipping','arrived','fast','slow','late'],Smell:['smell','odor','fragrance','scent','aroma','stink'],Service:['service','support','customer','return','refund']};

function setDemo(i){ document.getElementById('demoText').value=DEMOS[i]; }

async function runDemo(){
  const text=document.getElementById('demoText').value.trim();
  if(!text)return;
  const btn=document.getElementById('analyzeBtn');
  btn.disabled=true; btn.textContent='Analyzing...';
  try{
    const res  = await fetch('/analyze',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({text})});
    const data = await res.json();
    showResult(data,text);
  }catch(e){
    const low=text.toLowerCase();
    let ps=0,ns=0;
    posW.forEach(w=>{if(low.includes(w))ps++;});
    negW.forEach(w=>{if(low.includes(w))ns++;});
    const total=ps+ns;
    const score=total===0?0:(ps-ns)/(total+1);
    const conf=total===0?0.55:Math.min(0.97,0.5+Math.abs(score)*0.45);
    const sentiment=score>0.05?'Positive':score<-0.05?'Negative':'Neutral';
    const pros=[],cons=[];
    for(const[asp,kws]of Object.entries(ASPECTS)){
      if(kws.some(k=>low.includes(k))){
        const words=new Set(low.split(' '));
        if(posW.some(p=>words.has(p)))pros.push(asp);
        else if(negW.some(n=>words.has(n)))cons.push(asp);
      }
    }
    showResult({sentiment,score:score.toFixed(3),confidence:conf,pros,cons},text);
  }
  btn.disabled=false; btn.textContent='Analyze review';
}

function showResult(data,text){
  const isPos=data.sentiment==='Positive',isNeg=data.sentiment==='Negative';
  const bClass=isPos?'pos':isNeg?'neg':'neu';
  const fillColor=isPos?'#1D9E75':isNeg?'#D85A30':'#EF9F27';
  const emoji=isPos?'Positive':isNeg?'Negative':'Neutral';
  document.getElementById('resBadge').className='res-badge '+bClass;
  document.getElementById('resBadge').textContent=emoji;
  const fill=document.getElementById('confFill');
  fill.style.background=fillColor; fill.style.width='0%';
  setTimeout(()=>{fill.style.width=(data.confidence*100).toFixed(1)+'%';},50);
  document.getElementById('confLabel').textContent='Confidence: '+(data.confidence*100).toFixed(1)+'%';
  document.getElementById('resPros').innerHTML=data.pros.length?data.pros.map(p=>'<span class="tag pro">+ '+p+'</span>').join(''):'';
  document.getElementById('resCons').innerHTML=data.cons.length?data.cons.map(c=>'<span class="tag con">- '+c+'</span>').join(''):'';
  const sw=new Set(['the','a','an','is','was','were','be','have','has','do','will','this','that','it','i','we','you','and','or','but','in','on','to','for','of','with','very','so','too']);
  const tokens=text.toLowerCase().replace(/[^a-z\s]/g,'').split(' ').filter(w=>w&&!sw.has(w));
  document.getElementById('resTrace').innerHTML=
    '<div class="trace-row"><div class="tlabel">tokens</div><div class="tval">'+tokens.slice(0,10).map(t=>'<code>'+t+'</code>').join(' ')+(tokens.length>10?' <span style="color:var(--txt2);font-size:11px;">+'+(tokens.length-10)+' more</span>':'')+'</div></div>'+
    '<div class="trace-row"><div class="tlabel">pos signals</div><div class="tval" style="color:#0F6E56;">+'+posW.filter(w=>text.toLowerCase().includes(w)).length+' ('+( posW.filter(w=>text.toLowerCase().includes(w)).join(', ')||'none')+')</div></div>'+
    '<div class="trace-row"><div class="tlabel">neg signals</div><div class="tval" style="color:#4A1B0C;">-'+negW.filter(w=>text.toLowerCase().includes(w)).length+' ('+(negW.filter(w=>text.toLowerCase().includes(w)).join(', ')||'none')+')</div></div>'+
    '<div class="trace-row"><div class="tlabel">score</div><div class="tval" style="font-family:var(--mono);">'+(data.score>=0?'+':'')+data.score+'</div></div>'+
    '<div class="trace-row"><div class="tlabel">verdict</div><div class="tval" style="font-weight:500;">'+emoji+' - '+(data.confidence*100).toFixed(1)+'% confidence</div></div>';
  document.getElementById('resultBox').classList.add('show');
}

const STEPS=[
  {icon:'📄',name:'Raw input',tag:'stage 1',title:'Stage 1 - Raw input',desc:'The original review text is fed into the pipeline without any modifications.',ex:'Input: "This product is AMAZING!! Best quality ever...\\nhttps://amazon.com\\nScore: 5"'},
  {icon:'🧹',name:'Normalize',tag:'regex',title:'Stage 2 - Text normalization',desc:'Regular expressions clean the text: lowercase, strip HTML, remove URLs and special characters.',ex:'Before: "This product is AMAZING!!\\nhttps://amazon.com"\\nAfter:  "this product is amazing"'},
  {icon:'✂️',name:'Tokenize',tag:'nltk',title:'Stage 3 - Tokenization',desc:"NLTK word_tokenize splits the cleaned string into individual word tokens.",ex:"Input:  'this product is amazing best quality'\\nTokens: ['this','product','is','amazing','best','quality']"},
  {icon:'🚫',name:'Stopwords',tag:'filter',title:'Stage 4 - Stopword removal',desc:"Common English words are removed using NLTK stopwords corpus.",ex:"Before: ['this','product','is','amazing','best']\\nAfter:  ['product','amazing','best','quality']"},
  {icon:'🌱',name:'Stemming',tag:'porter',title:'Stage 5 - Stemming',desc:'Words reduced to root form using Porter Stemmer.',ex:'packaging   -> packag\\nrecommended -> recommend\\namazing     -> amaz'},
  {icon:'🏷️',name:'POS tag',tag:'syntax',title:'Stage 6 - POS Tagging',desc:'NLTK assigns grammatical categories to each token.',ex:'product -> NN  (noun)\\namazing -> JJ  (adjective)\\nbest    -> JJS (superlative)'},
  {icon:'📊',name:'Vectorize',tag:'BoW',title:'Stage 7 - Bag of Words',desc:'CountVectorizer converts text into a 5000-dimensional numerical feature vector.',ex:'Vocab: {product:0, amaz:1, best:2}\\nVector: product:1, amaz:1, best:1\\n(all other 4997 dims = 0)'},
  {icon:'🤖',name:'Classify',tag:'naive bayes',title:'Stage 8 - Naive Bayes',desc:'MultinomialNB computes probabilities for Positive/Negative/Neutral.',ex:'P(Positive|features) = 0.91\\nP(Negative|features) = 0.06\\nP(Neutral |features) = 0.03\\nResult: Positive'}
];

const bar=document.getElementById('pipelineBar');
bar.innerHTML=STEPS.map((s,i)=>'<div class="pipe-step'+(i===0?' active':'')+'" onclick="showStep('+i+')"><div class="pipe-icon">'+s.icon+'</div><div class="pipe-name">'+s.name+'</div><div class="pipe-tag">'+s.tag+'</div></div>').join('');
showStep(0);

function showStep(i){
  document.querySelectorAll('.pipe-step').forEach((s,idx)=>s.classList.toggle('active',idx===i));
  const s=STEPS[i];
  document.getElementById('stepTitle').textContent=s.title;
  document.getElementById('stepDesc').textContent=s.desc;
  document.getElementById('stepEx').textContent=s.ex;
}
</script>
</body>
</html>
"""

# ── Flask routes ─────────────────────────────────────────────
@app.route('/')
def index():
    html = HTML_PAGE
    html = html.replace('__SUMMARY_DATA__', json.dumps(summary_data))
    html = html.replace('__TOTALS__', json.dumps({
        'total': len(df),
        'pos':   total_pos,
        'neg':   total_neg,
        'neu':   total_neu
    }))
    html = html.replace('__ACCURACY__', str(acc_pct))
    return html

@app.route('/analyze', methods=['POST'])
def analyze():
    data       = request.get_json()
    text       = data.get('text','')
    score      = sia.polarity_scores(str(text))['compound']
    if score >= 0.05:    sentiment = 'Positive'
    elif score <= -0.05: sentiment = 'Negative'
    else:                sentiment = 'Neutral'
    confidence = min(0.99, 0.5 + abs(score)*0.5)
    pros, cons = extract_aspects(text)
    return jsonify({
        'sentiment':  sentiment,
        'score':      round(score,3),
        'confidence': round(confidence,3),
        'pros':       pros,
        'cons':       cons
    })

# ── Launch ────────────────────────────────────────────────────
ngrok.kill()

ngrok.set_auth_token("3ByXxP0lswhDI2qyL0NHQb07iOy_3u2JRAkhTAxRpdJ9WWngD")

thread = threading.Thread(target=lambda: app.run(port=5000, use_reloader=False))
thread.daemon = True
thread.start()

time.sleep(2)

public_url = ngrok.connect(5000)
print('\n' + '='*55)
print('  YOUR WEBSITE IS LIVE!')
print('='*55)
print(f'  Open this link: {public_url}')
print('='*55)
print('  Share this link to demo your project to anyone!')
print('  Link stays active as long as Colab is running\n')

 * Serving Flask app '__main__'
 * Debug mode: off


<>:447: SyntaxWarning: invalid escape sequence '\s'
<>:447: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_475/2803699559.py:447: SyntaxWarning: invalid escape sequence '\s'
  const tokens=text.toLowerCase().replace(/[^a-z\s]/g,'').split(' ').filter(w=>w&&!sw.has(w));
Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.



  YOUR WEBSITE IS LIVE!
  Open this link: NgrokTunnel: "https://effetely-unworthy-dwayne.ngrok-free.dev" -> "http://localhost:5000"
  Share this link to demo your project to anyone!
  Link stays active as long as Colab is running

